# 🔍🔍🔍 <span style="color: white; background-color: steelblue;"><b> Carga do Relatório do Esocial </b></span></p>

🧩 <span style="color: mediumseagreen;"><b> 1- Controle e Governança do Processo </b></span></p>
- ID do processo  
- Data/hora de início  
- Etapa 0
- Tudo no arquivo PROCESSOS.xlsx (aba REGISTROS)
- Esse registro segue o padrão das suas automações: total rastreabilidade e histórico completo de execuções

📥 <span style="color: mediumseagreen;"><b> 2- Identificação dos Arquivos do eSocial (Download) </b></span></p>
- A automação busca na pasta C:\\Users\\rodrigo.bernandes\\Downloads
- Todos os arquivos cujo nome começa com “Relacao de trabalhadores – eSocial”
- Para cada arquivo encontrado:
    - Extrai a data do arquivo diretamente do nome (8 dígitos)  
    - Ordena cronologicamente  
    - Inicia o processamento no sequenciamento adequado

🔧 <span style="color: mediumseagreen;"><b> 3- Carregamento e Preparação dos Dados </b></span></p>
- Lê o Excel bruto  
- Converte datas e campos problemáticos  
- Cria colunas auxiliares quando necessário  
- Aplica deduplicação inteligente por CPF, mantendo o registro mais atual (maior data de admissão)
- Deduplicação avançada
- A regra aplicada ordena por:
    - cpf  
    - matricula  
    - data_admissao (desc)
- Em seguida, remove duplicatas mantendo a versão mais recente por CPF

🧼 <span style="color: mediumseagreen;"><b> 4- Padronização completa com mapa de 84 colunas </b></span></p>
- Este processo possui um dos mapeamentos mais extensos entre suas automações:
    - 84 colunas mapeadas do eSocial → nomes padronizados internos
- Esse mapa garante padronização total entre diferentes fontes

📊 <span style="color: mediumseagreen;"><b> 5- Geração do Excel Formatado (TableStyleLight13) </b></span></p>
- Cria o arquivo final ESOCIAL.xlsx
- Com:
    - Tabela estruturada openpyxl  
    - Estilo TableStyleLight13  
    - Organização visual corporativa  
    - Colunas padronizadas e ordenadas
- Esse arquivo é sua base oficial tratada para eSocial

📦 <span style="color: mediumseagreen;"><b> 6- Movimentação dos Arquivos Originais </b></span></p>
- Cada arquivo processado é movido para ARQUIVOS MOVIDOS
- Mantendo:
    - Histórico  
    - Trilha de auditoria  
    - Limpeza no diretório de origem

🔄 <span style="color: mediumseagreen;"><b> 7- Atualização Automática do Controle HC e Atestados </b></span></p>
- O pipeline abre Controle_HC e Atestados.xlsb
- E usa PyAutoGUI para:
    - Navegar até a aba/célula correta  
    - Executar ALT + F5 para atualizar  
    - Salvar automaticamente
- Assim, os relatórios internos passam a refletir imediatamente os novos dados do eSocial

🗃️ <span style="color: mediumseagreen;"><b> 8- Geração do CSV para Banco de Dados </b></span></p>
- Converte ESOCIAL.xlsx → tb_esocial.csv
- Esse arquivo é utilizado em:
    - SQL  
    - Data Lake  
    - ETL  
    - Power BI  
    - Modelos analíticos

🧾 <span style="color: mediumseagreen;"><b> 9- Finalização e Resumo da Execução </b></span></p>
- O script:
    - Registra Etapa Final  
    - Exibe no terminal:tempo de execução  
    - Total de arquivos processados  
    - Total de registros  
    - Nome do arquivo gerado

# Importação das Bibliotecas

In [1]:
import duckdb as db
import os
import pandas as pd
import pyautogui
import shutil
import time
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from datetime import datetime, date
from openpyxl import Workbook, load_workbook

# Carregamento da base de controle de processos

In [2]:
tempo_0 = [id, datetime.today(), 0]

id = 17

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 0
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Variáveis com os Diretórios

In [3]:
# Diretório onde são salvos os arquivos extraídos
diretorio_buscar = r'C:\Users\rodrigo.bernandes\Downloads'

# Diretório onde são movidos os arquivos após a geração da base tratada
diretorio_mover = r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'

# Diretório onde será salvo o arquivo final
caminho_arquivo_final = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ESOCIAL.xlsx'

# Dataframe com o nome dos arquivos e data de extração

In [4]:
lista_arquivos = os.listdir(diretorio_buscar)
df_arquivos = pd.DataFrame(lista_arquivos, columns=['arquivo'])

# Filtra pelo prefixo
df_arquivos = df_arquivos[df_arquivos['arquivo'].str.startswith('Relacao de trabalhadores - eSocial')].copy()

# Extrai exatamente 8 dígitos contínuos (\d{8}) de qualquer lugar do nome do arquivo
datas_extraidas = df_arquivos['arquivo'].str.extract(r'(\d{8})')[0]

# Converte para datetime
df_arquivos['Data'] = pd.to_datetime(datas_extraidas, format='%Y%m%d', errors='coerce')

# Ordena e reseta o índice
df_arquivos = df_arquivos.sort_values(by='Data', ascending=True).reset_index(drop=True)

# Processo de Carga

In [5]:
MAPEAMENTO_COLUNAS = {
"Tipo de Inscrição do Empregador": "tipo_inscr_empregador",
"Número de Inscrição do Empregador": "num_inscr_empregador",
"Nome do Trabalhador": "nome",
"CPF do Trabalhador": "cpf",
"Matrícula": "matricula",
"Data de Admissão": "data_admissao",
"Código do Desligamento": "cod_desligamento",
"Motivo do Desligamento": "descricao_rescisao",
"Data do Efetivo Retorno da Reintegração": "data_retorno_reint",
"Data dos Efeitos Financeiros da Reintegração": "data_financ_reint",
"Regime Trabalhista": "regime_trabalhista",
"Regime Previdenciário": "regime_previdenciario",
"Natureza da Atividade": "natureza_atividade",
"Código do Tipo de Admissão": "cod_admissao",
"Tipo de Admissão": "tipo_admissao",
"Código do Tipo de Provimento": "cod_provimento",
"Tipo de Provimento": "tipo_provimento",
"Indicativo de Admissão": "indicativo_admissao",
"Código da Categoria": "cod_categoria",
"Categoria": "categoria",
"Cargo": "cargo",
"CBO do Cargo": "cbo_cargo",
"Função": "funcao",
"CBO da Função": "cbo_funcao",
"Tipo de Inscrição do Estabelecimento": "tipo_inscr_estabelec",
"Número de Inscrição do Estabelecimento": "num_inscr_estabelec",
"Quantidade média de horas semanais": "qtd_horas_semanais",
"Regime de Jornada": "regime_jornada",
"Salário Base": "salario",
"Código da Unidade de Pagamento": "cod_unid_pagamento",
"Unidade de Pagamento": "unidade_pagamento",
"Descrição do Salário Variável": "descricao_salario_variavel",
"Data da Última Alteração Contratual": "ultima_alteracao_contratual",
"Data dos Efeitos Remuneratorios da Última Alteracao Contratual": "dt_remun_ult_alt_contr",
"Data de Transferência": "data_transferencia",
"Tipo de Inscrição do Empregador Sucedido": "tipo_inscr_empreg_sucedido",
"Número de Inscrição do Empregador Sucedido": "num_inscr_empreg_sucedido",
"Matrícula no Empregador Sucedido": "matricula_empreg_sucedido",
"CPF do Empregador Doméstico Substituído": "cpf_empreg_domestico_subst",
"CPF Anterior do Trabalhador": "cpf_anterior_trabalhador",
"Tipo de Inscrição do Tomador de Trabalho Temporário": "tipo_inscr_tomador_trab_temp",
"Número de Inscrição do Tomador de Trabalho Temporário": "num_inscr_tomador_trab_temp",
"Código da Etnia e Raça": "cod_etnica_raca",
"Etnia e Raça": "etnica_raca",
"Sexo": "sexo",
"Data de Nascimento": "data_nascimento",
"Código do País de Nascimento": "cod_pais_nascimento",
"Código do País de Nacionalidade": "cod_pais_nacionalidade",
"Tempo de Residência do Imigrante": "tempo_residencia_imigrante",
"Condição de Ingresso do Imigrante": "cond_ingresso_imigrante",
"Deficiência Auditiva": "deficiencia_auditiva",
"Deficiência Física": "deficiencia_fisica",
"Deficiência Intelectual": "deficiencia_intelectual",
"Deficiência Mental": "deficiencia_mental",
"Deficiência Visual": "deficiencia_visual",
"Trabalhador Reabilitado ou Readaptado": "trab_reabilitado",
"Contabilizado para cota de PCD": "cota_pcd",
"Data da Última Alteração Cadastral": "data_ult_alter_cadastral",
"Modalidade de Contratação do Aprendiz": "modalidade_contrat_aprendiz",
"CNPJ da Entidade Qualificadora do Aprendiz na Modalidade Direta": "cnpj_entidade_aprendiz",
"Tipo de Inscrição do Estabelecimento Cumpridor da Cota de Aprendizagem na Modalidade Indireta": "tip_aprend_indireta",
"Número de Inscrição do Estabelecimento Cumpridor da Cota de Aprendizagem na Modalidade Indireta": "num_aprend_indireta",
"CNPJ do Estabelecimento das Atividades Práticas do Aprendiz": "cnpj_estabelecimento_aprendiz",
"Processo Trabalhista Indicado na Admissão": "processo_trab_admissao",
"Processo Trabalhista Indicado no Desligamento": "processo_trab_desligamento",
"Processo Trabalhista Indicado na Reintegração": "processo_trab_reintegracao",
"Processo Trabalhista Indicado na Baixa Judicial": "processo_trab_baixa_judic",
"Processo Trabalhista Indicado no Início de TSVE": "processo_trab_inicio_tsve",
"Processo Trabalhista Indicado no Término de TSVE": "processo_trab_termino_tsve",
"Número do Processo Indicado no evento de Reclamatória Trabalhista": "num_processo_trabalhista",
"Processo Trabalhista Indicado na Anotação Judicial do Vínculo": "processo_trabalhista",
"Tipo do Evento Inicial Vigente": "tipo_evento_inicial_vigente",
"Ambiente de Emissão do Evento Inicial Vigente": "ambiente_evento_inicial_vigente",
"Data de Envio da Admissão Preliminar Original": "data_admissao_prelim_orig",
"Número do Recibo da Admissão Preliminar Original": "num_recib_admissao_prelim_orig",
"Data de Envio da Admissão Original": "data_admissao_orig",
"Número do Recibo da Admissão Original": "num_recib_admissao_orig",
"Data de Envio do Início de TSVE Original": "data_inicio_tsve_orig",
"Número do Recibo do Início de TSVE Original": "num_recib_inicio_tsve_orig",
"Data de Envio do Evento de Processo Trabalhista Original": "data_event_proc_trab_orig",
"Número do Recibo do Evento de Processo Trabalhista Original": "num_recib_proc_trab_orig",
"Data de Envio da Anotação Judicial Original": "data_anot_judic_orig",
"Número do Recibo da Anotação Judicial Original": "num_recib_anot_judic_orig",
"Data de Envio da última Retificação do Evento de Admissão": "data_ult_retif_admissao",
"Número do Recibo da Última Retificação do Evento de Admissão": "num_recib_ult_retif_admissao",
"Data de Envio da Última Alteração Cadastral": "data_ult_alter_cadast",
"Número do Recibo da Última Alteração Cadastral": "num_recib_ult_alter_cadast",
"Data de Envio da Última Alteração Contratual": "data_ult_alter_contrat",
"Número do Recibo da última Alteração Contratual": "num_recib_ult_alter_contrat",
"Data de Envio do Desligamento": "data_desligamento",
"Número do Recibo do Desligamento": "num_recib_deslig",
"Data de Envio da última Retificação do Evento de Desligamento": "data_ult_retif_deslig",
"Número do Recibo da Última Retificação do Evento de Desligamento": "num_ult_retif_deslig",
"Data de Envio da Reintegração": "data_reintegracao",
"Número do Recibo da Reintegração": "num_recib_reintegracao",
"Data de Envio da última Retificação do Evento de Reintegração": "data_ult_retif_reint",
"Número do Recibo da Última Retificação do Evento de Reintegração": "num_recib_ult_retif_reint"
}

for a in df_arquivos['arquivo'].tolist():
    arquivo = os.path.join(diretorio_buscar, a)# Carregar os dados
    colaboradores = pd.read_excel(arquivo)

# 1. Preparar a coluna de Data de Admissão para a ordenação (Tratamento do CASE WHEN do SQL)
# Converte para datetime e preenche os nulos com '2050-12-31'
    colaboradores['Data de Admissão'] = pd.to_datetime(colaboradores['Data de Admissão'], errors='coerce')
    colaboradores['temp_data_admissao'] = colaboradores['Data de Admissão'].fillna(pd.Timestamp('2050-12-31'))

    # 2. Ordenação (Equivalente ao ORDER BY dentro do PARTITION BY)
    # CPF (agrupamento), Matrícula (ASC), Data Admissão (DESC)
    colaboradores = colaboradores.sort_values(
        by=['CPF do Trabalhador', 'Matrícula', 'temp_data_admissao'],
        ascending=[True, True, False]
    )

    # 3. Deduplicação (Equivalente ao ROW_NUMBER() = 1)
    # Mantém apenas o primeiro registro de cada CPF após a ordenação
    colaboradores = colaboradores.drop_duplicates(subset=['CPF do Trabalhador'], keep='first')

    # 4. Limpeza e Renomeação (Equivalente ao SELECT principal)
    # Renomeia as colunas usando o dicionário
    colaboradores = colaboradores.rename(columns=MAPEAMENTO_COLUNAS)

    # Filtra o DataFrame apenas para as colunas mapeadas (garante que colunas extras sejam descartadas)
    colunas_finais = [col for col in MAPEAMENTO_COLUNAS.values() if col in colaboradores.columns]
    colaboradores = colaboradores[colunas_finais]

    # =========================================================================
    # Geração do Arquivo Excel
    # =========================================================================
    wb = Workbook()
    ws = wb.active
    ws.title = "ESOCIAL"

    # Inserir dados no Excel
    for r in dataframe_to_rows(colaboradores, index=False, header=True):
        ws.append(r)

    # Formatar como Tabela
    tabela_colaboradores = Table(displayName="ESOCIAL", ref=ws.dimensions)
    estilo_tabela = TableStyleInfo(
        name="TableStyleLight13", 
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela_colaboradores.tableStyleInfo = estilo_tabela
    ws.add_table(tabela_colaboradores)

    # Salvar o arquivo processado
    wb.save(caminho_arquivo_final)  
                                    
    # Mover o arquivo original para a pasta de tratados
    destino_tratados = os.path.join(diretorio_mover, a)
    shutil.move(arquivo, destino_tratados)

# Atualizando o arquivo Excel Controle HC e Atestados

In [6]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo pelo Windows
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('ESOCIAL!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando base em CSV para o Banco de Dados

In [7]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\ESOCIAL.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_esocial.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [8]:
tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:01:34.046591

----------------------------------------------------------------------------------------------------
